In [10]:
from ..RAG import *
#pdf加载器
#使用pypdfLoader,一般需要结合其他依赖才能实现图像提取和保存图像，默认提取文本pdf
def save_images(image_data):
    """自定义图片解析器：将图片保存为文件"""
    for i, img_blob in enumerate(image_data):
        # img_blob 是图片的二进制数据（bytes）
        with open(f"../images/image_{i}", "wb") as f:
            f.write(img_blob)
        print(f"已保存图片: image_{i}.png")
pdf_loader=PyPDFLoader(
    file_path='../source/test.pdf',
    #提取模式为plain和layout,前者为默认值用于提取文本，后者布局感知，通过插入大量的空格、换行符，来模拟原文档中的多栏、缩进和间距
    extraction_mode='layout'
    #下面几个参数用于图片型pdf，加载图片
    #extract_images为是否开启图片提取模式
    # extract_images=True,
    #图片二进制解析器
    # images_parser=save_images,
    #用于表示解析后图像内容的输出格式，三种text,markdown-img,html-img,第一种原样返回内容，第二种将内容包装成 Markdown 图像链接（链接指向 ![body](#)），第三种将内容包装成 HTML 的 <img> 标签的 alt 文本，并链接到 <img alt="{body}" src="#"/>
    # images_inner_format='html-img'
)
docs=pdf_loader.load()
print(len(docs))

1


In [5]:
#非结构化加载器
#以word为例
word_loader=UnstructuredWordDocumentLoader(
    file_path='../source/test.docx',
    #提取模式分为single和elements，前者返回单个Document对象，后者按标题等元素切分文档
    mode='elements',
)
for doc in word_loader.lazy_load():
    print(doc.page_content[:10])

KeyboardInterrupt: 

In [8]:
#加载文件夹下指定文件
directory_loader=DirectoryLoader(
    path='../message',#文件路径
    glob='*.py',#加载文件路径下指定文件
    use_multithreading=True,#使用多线程，可以同时加载多个文件
    show_progress=True,#显示进度条
    loader_cls=PythonLoader,#实际上的文件加载器，加载.py就使用python加载器，默认为非结构文件加载器
)
docs=directory_loader.load()
print(type(docs[0]))
print(len(docs))
for doc in docs:
    pprint(doc.page_content)

100%|██████████| 2/2 [00:00<00:00, 634.35it/s]

<class 'langchain_core.documents.base.Document'>
2
('from  message import *\n'
 '\n'
 '\n'
 'def keep_recent_messages(messages, max_pairs=3):\n'
 '    """\n'
 '    保留最近的 N 轮对话（每轮 = user + assistant 或 Human + AI）。\n'
 '    支持字典格式和 LangChain BaseMessage 格式。\n'
 '    返回与输入相同类型的列表。\n'
 '    """\n'
 '    if not messages:\n'
 '        return []\n'
 '\n'
 '    # 检测类型：如果第一个元素是 BaseMessage 的子类，则按 BaseMessage 处理\n'
 '    is_base = isinstance(messages[0], BaseMessage)\n'
 '\n'
 '    if is_base:\n'
 '        # 处理 BaseMessage\n'
 '        system_msgs = [m for m in messages if isinstance(m, SystemMessage)]\n'
 '        conversation_msgs = [m for m in messages if not isinstance(m, '
 'SystemMessage)]\n'
 '        recent_msgs = conversation_msgs[-(max_pairs * 2):]\n'
 '        return system_msgs + recent_msgs\n'
 '    else:\n'
 '        # 处理字典格式\n'
 '        system_msgs = [m for m in messages if m.get("role") == "system"]\n'
 '        conversation_msgs = [m for m in messages if m.get("role") != '
 '"s